In [17]:
import pandas as pd


In [18]:
df= pd.read_csv('train.txt', sep=';',header= None, names=['text','emotion'])

In [19]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [20]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [21]:
unique_emotions= df['emotion'].unique()
emotion_numbers= {}
i=0
for emo in unique_emotions:
    emotion_numbers[emo]=i
    i+=1

df['emotion']= df['emotion'].map(emotion_numbers)

In [22]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [23]:
df['text']= df['text'].str.lower()

In [24]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [25]:
import string
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [26]:
df['text']= df['text'].apply(remove_punc)

In [27]:
def remove_numbers(txt):
  new=""
  for i in txt:
    if not i.isdigit():
      new = new + i
  return new

In [28]:
df['text']= df['text'].apply(remove_numbers)

In [29]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [30]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [31]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Uday\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Uday\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [32]:
stop_words = set(stopwords.words('english'))


In [33]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [34]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)

In [35]:
df['text'] = df['text'].apply(remove)

In [36]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [37]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [38]:
X = df['text']
y = df['emotion']

In [39]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], stratify=y, test_size=0.2, random_state=42)

In [40]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


In [41]:
bow_vectorizer= CountVectorizer()
tfidf_vectorizer= TfidfVectorizer()

In [42]:
X_train_bow= bow_vectorizer.fit_transform(X_train)
X_train_tfidf= tfidf_vectorizer.fit_transform(X_train)

In [43]:
X_test_bow= bow_vectorizer.transform(X_test)
X_test_tfidf= tfidf_vectorizer.transform(X_test)

In [44]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, f1_score

In [45]:
nb_model= MultinomialNB()

In [46]:
nb_model.fit(X_train_bow,y_train)



MultinomialNB()

In [47]:
y_pred= nb_model.predict(X_test_bow)

In [48]:
print(accuracy_score(y_test,y_pred))

0.7675


In [49]:
nb2_model= MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [50]:
y_pred= nb2_model.predict(X_test_tfidf)
print(accuracy_score(y_test,y_pred))

0.6621875


In [51]:
from sklearn.linear_model import LogisticRegression

In [52]:
log_model= LogisticRegression(max_iter=1000)

In [53]:
log_model.fit(X_train_bow,y_train)

LogisticRegression(max_iter=1000)

In [54]:
y_pred= log_model.predict(X_test_bow)
print(accuracy_score(y_test,y_pred))

0.886875


In [55]:
log_model2= LogisticRegression(max_iter=1000)
log_model2.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [56]:
y_pred= log_model2.predict(X_test_tfidf)
print(accuracy_score(y_test,y_pred))

0.8509375


In [57]:
import pickle

# Save trained logistic regression model
pickle.dump(log_model, open("emotion_model.pkl", "wb"))

# Save BOW vectorizer
pickle.dump(bow_vectorizer, open("bow_vectorizer.pkl", "wb"))

# Save emotion label mapping
pickle.dump(emotion_numbers, open("emotion_mapping.pkl", "wb"))